# Reconstruccion de `data_miRNA.txt`

Este notebook **regenera desde cero** el archivo `data_miRNA.txt` que necesita
`generate_miRNA_seq_2d.ipynb` y que no venia incluido en el proyecto.

`data_miRNA.txt` es una tabla con una fila por miRNA y estas columnas:
`Nombre miRNA, Secuencia miRNA Especifico, Nombre Precursor, Secuencia Precursor,
Secuencia 2D Precursor`.

Toda esa informacion (menos "Nombre Precursor", que el notebook no usa) esta en
`data/sequences_miRNA/`: la linea 1 de cada archivo es la secuencia del precursor
(con la region madura en MAYUSCULAS) y la linea 2 es la estructura 2D.

## Archivos de ENTRADA
- `data/sequences_miRNA/seq+ss_join_<miRNA>.txt`  -> linea 1 = precursor, linea 2 = 2D

## Archivo de SALIDA
- `data/data_miRNA.txt`  (CSV con encabezado, separado por comas)

## Como se obtiene cada columna
- **Nombre miRNA**            -> del nombre del archivo
- **Secuencia miRNA Especifico** -> las letras en MAYUSCULA del precursor (region madura)
- **Nombre Precursor**        -> no se puede recuperar; se pone "NA" (el notebook no lo usa)
- **Secuencia Precursor**     -> linea 1 del archivo
- **Secuencia 2D Precursor**  -> linea 2 del archivo


In [1]:
import os
import csv

# ====================== CONFIGURACION ======================
SEQ_MIR_DIR    = "./../data/sequences_miRNA"   # archivos seq+ss_join_*.txt de miRNA
OUT_DATA_MIRNA = "./../data/data_miRNA.txt"    # tabla reconstruida
# ============================================================

In [2]:
PREFIX = "seq+ss_join_"
COLUMNS = ["Nombre miRNA", "Secuencia miRNA Especifico",
           "Nombre Precursor", "Secuencia Precursor", "Secuencia 2D Precursor"]


def read_seq_ss(path):
    """Lee un archivo seq+ss_join_*.txt: linea 1 = precursor, linea 2 = estructura 2D."""
    with open(path) as f:
        lines = f.read().splitlines()
    precursor = lines[0].strip() if len(lines) > 0 else ""
    ss        = lines[1].strip() if len(lines) > 1 else ""
    return precursor, ss


# Recorrer la carpeta y armar una fila por miRNA
rows = []
for fname in sorted(os.listdir(SEQ_MIR_DIR)):
    if not fname.startswith(PREFIX):
        continue
    name = os.path.splitext(fname[len(PREFIX):])[0]
    precursor, ss = read_seq_ss(os.path.join(SEQ_MIR_DIR, fname))
    if not precursor or not ss:          # archivo incompleto: se omite
        continue
    mature = "".join(c for c in precursor if c.isupper())  # region madura
    rows.append([name, mature, "NA", precursor, ss])

print(f"miRNA procesados: {len(rows)}")

miRNA procesados: 2656


In [3]:
# Escribir data_miRNA.txt (CSV con encabezado)
with open(OUT_DATA_MIRNA, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(COLUMNS)
    writer.writerows(rows)

print(f"Archivo generado: {OUT_DATA_MIRNA}")

Archivo generado: ./../data/data_miRNA.txt


In [4]:
# --- Verificacion: leerlo con pandas, igual que generate_miRNA_seq_2d.ipynb ---
import pandas as pd

data_miRNA = pd.read_csv(OUT_DATA_MIRNA)
print("=== VERIFICACION ===")
print(f"filas: {len(data_miRNA)}")
print(f"columnas: {list(data_miRNA.columns)}")

faltan = [c for c in COLUMNS if c not in data_miRNA.columns]
if faltan:
    print(f"ATENCION: faltan columnas: {faltan}")
else:
    print("OK: estan las 5 columnas que espera generate_miRNA_seq_2d.ipynb")

ej = data_miRNA.iloc[0]
print(f"\nEjemplo  {ej['Nombre miRNA']}")
print(f"  madura    : {ej['Secuencia miRNA Especifico']}")
print(f"  precursor : {ej['Secuencia Precursor'][:60]}...")
print(f"  2D        : {ej['Secuencia 2D Precursor'][:60]}...")

=== VERIFICACION ===
filas: 2656
columnas: ['Nombre miRNA', 'Secuencia miRNA Especifico', 'Nombre Precursor', 'Secuencia Precursor', 'Secuencia 2D Precursor']
OK: estan las 5 columnas que espera generate_miRNA_seq_2d.ipynb

Ejemplo  hsa-let-7a-2-3p
  madura    : TGAGGTAGTAGGTTGTATAGTTCTGTACAGCCTCCTAGCTTTCC
  precursor : agguTGAGGTAGTAGGTTGTATAGTTuagaauuacaucaagggagauaaCTGTACAGCCT...
  2D        : (((..(((.(((.(((((((((((((.........(((......))))))))))))))))...
